# Aplicación: aprobación de créditos

## 2.1 Importar librerías
- `numpy`, `pandas`: datos numéricos y tablas.
- `matplotlib`: gráficos.
- `DecisionTreeClassifier`: árbol de decisión para clasificación.
- `plot_tree`, `export_text`: visualizar el árbol como figura o como reglas de texto.
- `RandomForestClassifier`: para comparar con un ensamble.
- `train_test_split`, `GridSearchCV`, `cross_val_score`: partición de datos y validación.
- `metrics`: evaluación.
- `joblib`: guardar el modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import joblib

## 2.2 Generar los datos de las solicitudes
Se **simulan** 2.000 solicitudes de crédito (en una aplicación real se reemplaza este bloque por la lectura de un archivo, por ejemplo `pd.read_csv('solicitudes.csv')`).
- `np.random.default_rng(42)`: generador de números aleatorios reproducible.
- Variables: `ingreso_mensual`, `edad`, `ratio_deuda` (deuda mensual / ingreso), `moras_ultimos_2_anios` y `antiguedad_laboral` (años).
- La variable objetivo `aprobado` (1 = sí, 0 = no) sigue reglas de negocio razonables, más un 5% de casos "ruidosos" (excepciones) para imitar la realidad.
- `np.clip`: limita los valores a un rango válido.

In [ ]:
rng = np.random.default_rng(42)
n = 2000

datos = pd.DataFrame({
    'ingreso_mensual': np.clip(rng.normal(3000, 1100, n), 800, None).round(0),
    'edad': rng.integers(18, 71, n),
    'ratio_deuda': np.clip(rng.uniform(0, 1.1, n), 0, None).round(2),
    'moras_ultimos_2_anios': rng.poisson(0.8, n),
    'antiguedad_laboral': np.clip(rng.exponential(6, n), 0, 35).round(1)
})

# Reglas de negocio (desconocidas para el modelo) + ruido
regla1 = (datos.ratio_deuda < 0.5) & (datos.moras_ultimos_2_anios <= 1) & (datos.ingreso_mensual > 1800)
regla2 = (datos.antiguedad_laboral > 10) & (datos.ratio_deuda < 0.65) & (datos.moras_ultimos_2_anios == 0)
datos['aprobado'] = (regla1 | regla2).astype(int)

ruido = rng.random(n) < 0.05
datos.loc[ruido, 'aprobado'] = 1 - datos.loc[ruido, 'aprobado']

print("Solicitudes por decisión (1 = aprobado):")
print(datos['aprobado'].value_counts())
datos.head()

## 2.3 Separar variables y entrenamiento/prueba
- `X`: variables de entrada; `y`: decisión.
- `train_test_split(test_size=0.25, stratify=y)`: 75% para entrenar, 25% para probar, conservando la proporción de aprobados.
- **No se escala** ningún dato: los árboles no usan distancias, solo comparan umbrales.

In [ ]:
X = datos.drop(columns='aprobado')
y = datos['aprobado']
columnas = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

## 2.4 Árbol sin límites: el problema del sobreajuste
- `DecisionTreeClassifier(random_state=42)`: sin restricciones crece hasta separar todo.
- `get_depth()` y `get_n_leaves()`: profundidad y número de hojas.
- Se compara la precisión en entrenamiento y en prueba: una gran diferencia indica sobreajuste.

In [ ]:
arbol_libre = DecisionTreeClassifier(random_state=42)
arbol_libre.fit(X_train, y_train)

print(f"Profundidad: {arbol_libre.get_depth()} | Hojas: {arbol_libre.get_n_leaves()}")
print(f"Precisión entrenamiento: {arbol_libre.score(X_train, y_train)*100:.2f}%")
print(f"Precisión prueba:        {arbol_libre.score(X_test, y_test)*100:.2f}%")

## 2.5 Efecto de la profundidad máxima
- Se entrenan árboles con `max_depth` de 1 a 15.
- `cross_val_score(cv=5)`: precisión en validación cruzada (estimación fiable de generalización).
- La curva de entrenamiento sube siempre, pero la de validación se estanca o baja: ahí empieza el sobreajuste.

In [ ]:
profundidades = range(1, 16)
acc_train, acc_val = [], []
for d in profundidades:
    arbol = DecisionTreeClassifier(max_depth=d, random_state=42)
    arbol.fit(X_train, y_train)
    acc_train.append(arbol.score(X_train, y_train))
    acc_val.append(cross_val_score(arbol, X_train, y_train, cv=5).mean())

plt.figure(figsize=(8, 4))
plt.plot(profundidades, acc_train, marker='o', label='Entrenamiento')
plt.plot(profundidades, acc_val, marker='o', label='Validación cruzada')
plt.xlabel('Profundidad máxima')
plt.ylabel('Precisión')
plt.title('Sobreajuste según la profundidad')
plt.legend()
plt.show()

## 2.6 Optimización con GridSearchCV (pre-poda)
- `param_grid`: combinaciones de `criterion`, `max_depth` y `min_samples_leaf`.
- `GridSearchCV`: prueba todas con validación cruzada y se queda con la mejor.
- `best_params_` / `best_estimator_`: mejores parámetros y modelo ya entrenado.

In [ ]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'min_samples_leaf': [1, 5, 10, 20]
}

busqueda = GridSearchCV(DecisionTreeClassifier(random_state=42),
                        param_grid, cv=5, scoring='accuracy', n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejores parámetros:", busqueda.best_params_)
print(f"Precisión en validación cruzada: {busqueda.best_score_*100:.2f}%")
mejor_arbol = busqueda.best_estimator_

## 2.7 Post-poda por costo-complejidad
- `cost_complexity_pruning_path`: calcula la secuencia de valores `ccp_alpha` con los que se va podando el árbol completo.
- Para algunos valores de alpha se entrena un árbol y se mide su precisión en validación cruzada.
- Se escoge el alpha con mejor precisión; un alpha mayor produce un árbol más pequeño y simple.

In [ ]:
camino = DecisionTreeClassifier(random_state=42).cost_complexity_pruning_path(X_train, y_train)
alphas = camino.ccp_alphas[:-1]                      # se descarta el árbol vacío
alphas = alphas[np.linspace(0, len(alphas) - 1, 30).astype(int)]

acc_alpha = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=42),
                             X_train, y_train, cv=5).mean() for a in alphas]
mejor_alpha = alphas[int(np.argmax(acc_alpha))]

plt.figure(figsize=(8, 4))
plt.plot(alphas, acc_alpha, marker='o')
plt.axvline(mejor_alpha, color='red', linestyle='--', label=f'Mejor alpha = {mejor_alpha:.4f}')
plt.xlabel('ccp_alpha')
plt.ylabel('Precisión (validación cruzada)')
plt.title('Poda por costo-complejidad')
plt.legend()
plt.show()

arbol_podado = DecisionTreeClassifier(ccp_alpha=mejor_alpha, random_state=42).fit(X_train, y_train)
print(f"Árbol podado -> profundidad: {arbol_podado.get_depth()}, hojas: {arbol_podado.get_n_leaves()}")
print(f"Precisión en prueba (podado): {arbol_podado.score(X_test, y_test)*100:.2f}%")

## 2.8 Evaluar el mejor árbol en los datos de prueba
- `predict()`: cada solicitud recorre el árbol desde la raíz hasta una hoja.
- `classification_report`: precisión, recall y F1 por clase.
- `confusion_matrix`: aciertos y errores. En crédito, aprobar a un mal pagador (falso positivo) suele ser más costoso que rechazar a un buen cliente.

In [ ]:
pred = mejor_arbol.predict(X_test)

print(f"Precisión en prueba: {accuracy_score(y_test, pred)*100:.2f}%\n")
print(classification_report(y_test, pred, target_names=['Rechazado', 'Aprobado'], digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=['Rechazado', 'Aprobado']).plot(cmap='Blues', colorbar=False)
plt.title('Matriz de confusión')
plt.show()

## 2.9 Visualizar el árbol y sus reglas
- `plot_tree`: dibuja el árbol. Cada nodo muestra la pregunta, el Gini/entropía, el número de muestras, cuántas hay de cada clase y la clase predominante. `filled=True` colorea según la clase.
- `export_text`: imprime el árbol como reglas *si... entonces...*, lista para documentar o revisar con un experto.
- `feature_importances_`: importancia de cada variable.

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(mejor_arbol, feature_names=columnas, class_names=['Rechazado', 'Aprobado'],
          filled=True, rounded=True, max_depth=3, fontsize=9)
plt.title('Árbol de decisión (primeros 3 niveles)')
plt.show()

print(export_text(mejor_arbol, feature_names=columnas, max_depth=3))

In [ ]:
importancias = pd.Series(mejor_arbol.feature_importances_, index=columnas).sort_values()
importancias.plot(kind='barh', figsize=(7, 3.5), color='teal')
plt.title('Importancia de las variables')
plt.xlabel('Importancia')
plt.show()

## 2.10 Comparación con un ensamble (Random Forest)
- `RandomForestClassifier(n_estimators=200)`: entrena 200 árboles sobre muestras aleatorias y promedia sus votos.
- Suele ser más preciso y estable, pero **pierde la explicación en reglas** de un único árbol.

In [ ]:
bosque = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
bosque.fit(X_train, y_train)

print(f"Árbol libre:    {arbol_libre.score(X_test, y_test)*100:.2f}%")
print(f"Árbol optimizado: {mejor_arbol.score(X_test, y_test)*100:.2f}%")
print(f"Árbol podado:   {arbol_podado.score(X_test, y_test)*100:.2f}%")
print(f"Random Forest:  {bosque.score(X_test, y_test)*100:.2f}%")

## 2.11 Convertir el modelo en una aplicación
La función `evaluar_solicitud` recibe los datos de un solicitante y devuelve la decisión **junto con las reglas que la justifican**:
1. `joblib.dump` / `joblib.load`: guardan y cargan el modelo.
2. `predict_proba`: probabilidad de aprobación según las muestras de la hoja.
3. `decision_path`: nodos por los que pasó la solicitud.
4. `tree_.feature` y `tree_.threshold`: variable y umbral de cada nodo, para mostrar cada pregunta y su respuesta.
5. Los nodos hoja se omiten al listar las preguntas.

In [ ]:
joblib.dump(mejor_arbol, 'modelo_arbol_credito.joblib')
modelo = joblib.load('modelo_arbol_credito.joblib')

def evaluar_solicitud(modelo, datos_solicitante):
    """Devuelve la decisión de crédito y explica el camino de reglas seguido."""
    fila = pd.DataFrame([datos_solicitante])[columnas]
    clase = modelo.predict(fila)[0]
    proba = modelo.predict_proba(fila)[0][1]

    print("DECISIÓN:", "APROBADO" if clase == 1 else "RECHAZADO",
          f"(probabilidad de aprobación: {proba*100:.0f}%)")
    print("Reglas aplicadas:")
    arbol = modelo.tree_
    for nodo in modelo.decision_path(fila).indices:
        if arbol.children_left[nodo] == arbol.children_right[nodo]:   # hoja
            continue
        variable = columnas[arbol.feature[nodo]]
        umbral = arbol.threshold[nodo]
        valor = fila.iloc[0][variable]
        signo = '<=' if valor <= umbral else '>'
        print(f"  - {variable} = {valor:g} {signo} {umbral:.2f}")
    return clase

solicitantes = {
    'Solicitante A (perfil sólido)': {'ingreso_mensual': 4500, 'edad': 38, 'ratio_deuda': 0.25,
                                      'moras_ultimos_2_anios': 0, 'antiguedad_laboral': 9},
    'Solicitante B (muy endeudado)': {'ingreso_mensual': 2800, 'edad': 30, 'ratio_deuda': 0.85,
                                      'moras_ultimos_2_anios': 2, 'antiguedad_laboral': 3},
    'Solicitante C (ingreso bajo)':  {'ingreso_mensual': 1200, 'edad': 45, 'ratio_deuda': 0.30,
                                      'moras_ultimos_2_anios': 0, 'antiguedad_laboral': 4},
}
for nombre, datos_s in solicitantes.items():
    print(f"=== {nombre} ===")
    evaluar_solicitud(modelo, datos_s)
    print()